In [42]:
import pandas as pd
import os
from collections import defaultdict,Counter
import regex as re
import heapq
from tqdm import tqdm

In [43]:
df=pd.read_parquet("../data/fineweb-edu-10BT/sample/10BT/000_00000.parquet")
texts=df['text'].tolist()

In [44]:
sample_text=texts[:100]

In [45]:
GPT2_SPLIT_PATTERN = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
GPT4_SPLIT_PATTERN = re.compile(r"""(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+""")


In [46]:
word_counts = Counter()
for text in sample_text:
	word_counts.update(re.findall(GPT2_SPLIT_PATTERN, text))

In [47]:
def get_word_counts(texts):
    GPT4_SPLIT_PATTERN = re.compile(r"""(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+""")
    word_counts = Counter()
    
    for text in tqdm(texts):
        word_counts.update(re.findall(GPT4_SPLIT_PATTERN, text))
    
    return word_counts

In [48]:
get_word_counts(texts[:10])

100%|██████████| 10/10 [00:00<00:00, 1982.09it/s]


Counter({',': 377,
         ' the': 331,
         '.': 242,
         ' of': 209,
         ' to': 190,
         ' a': 182,
         ' and': 174,
         '.\n': 110,
         ' in': 104,
         ' you': 97,
         ' ': 94,
         ' is': 83,
         ' for': 80,
         ' that': 60,
         ' software': 58,
         ' on': 51,
         "'s": 51,
         ' with': 50,
         ' or': 47,
         ' as': 43,
         ' (': 40,
         ' can': 39,
         ' are': 38,
         ' have': 34,
         ' at': 33,
         ' "': 33,
         ' more': 32,
         ':': 31,
         ' play': 31,
         ' licensing': 31,
         '\n': 29,
         ' by': 29,
         ' an': 28,
         ' from': 27,
         ' license': 27,
         ' membrane': 27,
         ' it': 26,
         ' server': 26,
         ' I': 25,
         ' not': 25,
         ' other': 25,
         ' all': 24,
         ' be': 24,
         ' your': 24,
         ' one': 23,
         '200': 23,
         ' licenses': 23,
     

In [ ]:
class WordNode():
    def __init__(self,token=None,next=None,prev=None,count=None):
        self.token=token
        self.next=next
        self.prev=prev
        self.count=count    
        
class WordList():
    def __init__(self,text=None,count=None,index_map=None):
        text=text.encode('utf-8')
        byte=list(bytes(text))
        self.head=None
        self.count=count
        self.head=WordNode((byte[0]),count=self.count)
        curr=self.head
        
        if len(byte)>1:
            i=0
            index_map[(byte[i],byte[i+1])]["pos"].append(self.head)
            index_map[(byte[i],byte[i+1])]["count"]+=count

            for i in range(1,len(byte)-1):
                node=WordNode((byte[i]),count=self.count)
                curr.next=node
                node.prev=curr
                curr=node
                
                index_map[(byte[i],byte[i+1])]["pos"].append(node)
                index_map[(byte[i],byte[i+1])]["count"]+=count
            
            node=WordNode((byte[len(byte)-1]),count=self.count)
            curr.next=node
            node.prev=curr
            curr=node
                            

class Index_Heap():
    def __init__(self,word_counts):
        self.heap=[]
        self.index_map=defaultdict(lambda: {"pos": [], "count": 0,"time":0})        
        self.wordchain=[]    
        self.counter=0
        self.time_hash={}
        self.modified=set()
        

        for text in word_counts:
            self.wordchain.append(WordList(text,word_counts[text],self.index_map))

        for token_pair in self.index_map:
            count=-self.index_map[token_pair]['count']
            self.heap.append((count,0,token_pair))

        heapq.heapify(self.heap)

    
    def heap_add(self,token_pair,count):
        self.counter+=1
        heapq.heappush(self.heap,(-count,-self.counter,token_pair))
        self.time_hash[token_pair]=-self.counter
        

    def merge(self,node,token_pair):
            token1,token2=token_pair
            if node.token==token1 and node.next.token==token2:
                node.token=(token1,token2)
                
                right_node=node.next
                

                if node.prev:
                    self.index_map[(node.prev.token,token1)]["count"]-=node.prev.count                    
                    self.index_map[(node.prev.token,node.token)]["pos"].append(node.prev)
                    self.index_map[(node.prev.token,node.token)]["count"]+=node.prev.count
                    self.modified.add((node.prev.token,node.token))
                    self.modified.add((node.prev.token,token1))
                
                if node.next.next:
                    self.index_map[(token2,node.next.next.token)]["count"]-=node.next.next.count                    
                    node.next.next.prev=node

                    self.index_map[(node.token,node.next.next.token)]["pos"].append(node)
                    self.index_map[(node.token,node.next.next.token)]["count"]+=node.next.next.count
                    self.modified.add((node.token,node.next.next.token))
                    self.modified.add((token2,node.next.next.token))
                
                node.next=node.next.next
                right_node.next=None
                right_node.prev=None

                return
            

    def get_max(self):
        while self.heap:
            
            count,time,token_pair=heapq.heappop(self.heap)          
            if count<0 and time==self.time_hash.get(token_pair,0):
                    return token_pair
        return None
    
    
    def update_heap(self):
        for token_pair in self.modified:
            self.heap_add(token_pair,self.index_map[token_pair]["count"])
    
    
    def update_index(self,token_pair):
        self.modified=set()

        for node in self.index_map[token_pair]["pos"]:
            if node.next and node.token==token_pair[0] and node.next.token==token_pair[1]:
                self.merge(node,token_pair)
        
        self.modified.discard(token_pair)
        self.update_heap()
        del self.index_map[token_pair]
        return

        

In [ ]:
def arr_from_tuples_rec(t):
    if type(t)==int:
        return [t]
    else:
        tuple1,tuple2=t
        return arr_from_tuples_rec(tuple1) + arr_from_tuples_rec(tuple2)

arr=arr_from_tuples_rec( (((104, 101), 108), ((108, 111), 32)))
bytes(arr),bytes([125])


(b'hello ', b'}')

In [ ]:
def build_vocab(texts,vocab_size=48000):
    word_counts=get_word_counts(texts)
    index_heap=Index_Heap(word_counts)    
    vocab={}

    for i in range(256):
        vocab[bytes([i])]=i	
    
    for j in tqdm(range(i+1,vocab_size)):
        maxm=index_heap.get_max()

        if maxm:
            maxm_bytes=bytes(arr_from_tuples_rec(maxm))
            vocab[maxm_bytes]=j
        else:
            break

        index_heap.update_index(maxm)

    return vocab        
        
        
vocab=build_vocab(texts[:10000])
print(len(vocab.keys()))
    

100%|██████████| 47734/47734 [00:07<00:00, 6332.87it/s] 


47980


In [52]:
import tiktoken
import base64

vocab
pat_str = r"""(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+"""

# 3. Create your custom Tiktoken Encoding
custom_encoding = tiktoken.Encoding(
    name="custom_encoding",
    pat_str=pat_str,
    mergeable_ranks=vocab,
    special_tokens={
        "<|endoftext|>": len(vocab)
    } # Add your model's special tokens here
)

text = "hello en"
tokens = custom_encoding.encode(text)
print("Token IDs:", tokens)

decoded_text = custom_encoding.decode(tokens)
print("Decoded Text:", decoded_text)


Token IDs: [35185, 473]
Decoded Text: hello en
